# Contextual bandits — interactive companion

Companion to [Post 1b: Contextual bandits](../posts/01b-contextual-bandits.qmd).
Same exploration question as bandits, now with a *context vector* observed
before each pull. The reward depends on both the context and the chosen arm.

**What you'll do (≈ 15 minutes):**
1. Generate a linear contextual bandit.
2. Compare LinUCB (knows the structure) against context-free baselines.
3. Switch to a nonlinear reward function and watch LinUCB break.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["figure.dpi"] = 110

from nano_agents.contextual import (
    LinearContextualBandit, NonlinearContextualBandit,
    LinUCB, ContextFreeUCB, ContextualEpsilonGreedy,
    run_many_contextual,
)

## 1. Linear contextual bandit

Each arm $k$ has a hidden coefficient vector $\theta_k \in \mathbb{R}^d$.
Before each pull, the world gives a context $x \in \mathbb{R}^d$.
The expected reward for arm $k$ is $\theta_k^\top x$. We see noisy
realizations of this and have to figure out which arm to pull.

LinUCB extends UCB1: each arm's reward is a *linear* function of the
context, fit by ridge regression. The confidence bound is the radius
of an ellipsoid around the estimate.

In [ ]:
def make_linear(seed):
    return LinearContextualBandit(K=5, d=4, seed=seed)

K, d = 5, 4
agents = {
    "LinUCB":                                lambda: LinUCB(K=K, d=d, alpha=1.0),
    r"contextual $\varepsilon$-greedy (0.1)": lambda: ContextualEpsilonGreedy(K=K, d=d, eps=0.1),
    "context-free UCB":                      lambda: ContextFreeUCB(K=K, d=d),
}
T, n_runs = 1500, 15
results = run_many_contextual(agents, make_linear, T=T, n_runs=n_runs)

for name, regrets in results.items():
    plt.plot(regrets, label=name)
plt.xlabel("step"); plt.ylabel(f"cumulative regret (mean over {n_runs} runs)")
plt.title(f"Linear contextual bandit (K=5 arms, d=4 dims)")
plt.legend(); plt.grid(alpha=0.3); plt.show()

LinUCB should beat both context-free baselines decisively — it can learn
*how the optimal arm depends on the context*, whereas the context-free
methods are stuck guessing the marginally-best arm.

### Try this
- Set `d=10`. With higher-dimensional contexts, LinUCB has more to learn but
  more to gain. Does the gap widen or narrow?
- Set `K=20`. Many arms with linear structure — LinUCB should still scale.
- Reduce `alpha=0.1` (LinUCB's exploration coefficient). What happens?

## 2. The linearity assumption breaks

LinUCB is *exactly* right when rewards are linear in the context. What
happens when they aren't? Let's use a sigmoid reward and watch LinUCB struggle.

In [ ]:
def make_nonlinear(seed):
    return NonlinearContextualBandit(K=5, d=4, seed=seed)

agents = {
    "LinUCB":                                lambda: LinUCB(K=K, d=d, alpha=1.0),
    r"contextual $\varepsilon$-greedy (0.1)": lambda: ContextualEpsilonGreedy(K=K, d=d, eps=0.1),
    "context-free UCB":                      lambda: ContextFreeUCB(K=K, d=d),
}
results = run_many_contextual(agents, make_nonlinear, T=T, n_runs=n_runs)

for name, regrets in results.items():
    plt.plot(regrets, label=name)
plt.xlabel("step"); plt.ylabel(f"cumulative regret (mean over {n_runs} runs)")
plt.title("Nonlinear (sigmoid) contextual bandit — model is misspecified")
plt.legend(); plt.grid(alpha=0.3); plt.show()

**Honest result.** LinUCB still does *something* — sigmoid is locally linear
near the decision boundary, and LinUCB picks up that approximation. But its
asymptotic regret is now linear (not logarithmic): it keeps making mistakes
forever because its model is biased.

This is the *model misspecification* problem in a nutshell. With perfect
linear structure, LinUCB is great. Without it, LinUCB is just "epsilon-greedy
with a fancy estimator."

The fix is usually one of two things:
- Use a richer function class (neural networks, Gaussian processes).
- Or use a method that doesn't require knowing the structure (Thompson sampling
  with non-parametric prior, ε-greedy with a neural network).

### Try this
- Add a `ContextualEpsilonGreedy` with `eps=0.05` and a *neural network* —
  oh wait, we don't have that here. But that's a great exercise. The
  `nano_agents.contextual.agents` module is short; copy `ContextualEpsilonGreedy`
  and swap in a 2-layer MLP for the linear predictor.
- Reduce LinUCB's `alpha` to 0.1. With less exploration, does it converge
  to a stable (but suboptimal) policy?

## What's next

Contextual bandits are *one-step* problems: you see a context, pull an arm,
get a reward, done. The next step adds time: each action changes the world,
and rewards may come later.

That's the MDP setting. Open
[`01c-mdps-and-bellman.ipynb`](01c-mdps-and-bellman.ipynb) for value
iteration and policy iteration on gridworlds.